# 06 · Congelar el protocolo antes de abrir el externo

Este cuaderno verifica que entrenamiento, estabilidad, controles, ablaciones y validación biológica estén completos. Exige explícitamente el consenso biológico bloqueado en el notebook 04 con pesos de contexto y soporte 0.35/0.075, comprueba que el notebook 05 utilizó los artefactos esperados y calcula SHA-256 de código, notebooks, entradas, modelos y resultados.

No analiza expresión ni etiquetas externas: únicamente fija sus bytes mediante hash. Si se cambia cualquier fichero después de crear el bloqueo, el notebook 07 se negará a evaluar. El notebook 07 debe ejecutarse una sola vez.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute este notebook dentro del repositorio InterGATE.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_v6_protocol, assert_transport_protocol, prepare_runtime,
)

assert_v6_protocol(CFG)
assert_transport_protocol(CFG)
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / "INTERGATE_V6"
RESULTS = PROJECT_ROOT / "results_v6"
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
import hashlib
from intergate.reviewer_analysis import freeze_protocol

# El externo debe permanecer cerrado durante todo el notebook 06.
runtime = prepare_runtime(CFG, include_external=False)
if runtime.get("external_opened") is not False:
    raise RuntimeError("El runtime abrió el bloque externo durante el congelado.")

LOCK = ARTIFACTS / "protocol_lock_v6.json"
if LOCK.exists():
    raise FileExistsError(
        f"Ya existe {LOCK}. No lo sobrescriba ni repita el congelado."
    )

# Selección estructural definitiva fijada en el notebook 04 antes de ejecutar
# la validación biológica del notebook 05 y antes de abrir el test externo.
LOCKED_BIOLOGY = RESULTS / "04_controls/locked_biology_score_w0075"
LOCKED_CONSENSUS = LOCKED_BIOLOGY / "biology_consensus"
LOCKED_MANIFEST = LOCKED_BIOLOGY / "locked_score_manifest.json"
LOCKED_PANELS = LOCKED_BIOLOGY / "biology_panels_locked.json"
DEPMAP_MANIFEST = RESULTS / "05_biology_validation/depmap_manifest.json"

required_results = [
    # Preflight, entrenamiento y estabilidad original (trazabilidad).
    RESULTS / "01_preflight/preflight_summary.json",
    RUN_ROOT / "development_results.csv",
    RESULTS / "03_stability/biology_consensus/biology_stability_summary.json",
    RESULTS / "03_stability/biology_panels_v6.json",

    # Baselines, controles y ablaciones atómicas.
    RESULTS / "04_controls/baseline_feature_universe_audit.csv",
    RESULTS / "04_controls/predictive_baselines_loco.csv",
    RESULTS / "04_controls/one_seed_context_ablation.csv",
    RESULTS / "04_controls/trained_ablations/single_component_ablation_contract.csv",
    RESULTS / "04_controls/trained_ablations/trained_ablation_predictive_metrics.csv",
    RESULTS / "04_controls/trained_ablations/trained_ablation_biology_Basal-like.csv",
    RESULTS / "04_controls/posthoc_score_sensitivity_Basal-like/posthoc_score_network_summary_Basal-like.csv",
    RESULTS / "04_controls/posthoc_score_sensitivity_Basal-like/posthoc_score_edge_overlap_Basal-like.csv",

    # Score biológico y consenso definitivamente bloqueados.
    LOCKED_MANIFEST,
    LOCKED_BIOLOGY / "locked_per_seed_network_summary.csv",
    LOCKED_CONSENSUS / "biology_stability_summary.json",
    LOCKED_PANELS,

    # Validación biológica y ortogonal completa.
    RESULTS / "05_biology_validation/matched_candidate_signature_tests.csv",
    RESULTS / "05_biology_validation/matched_candidate_signature_distribution.csv.gz",
    RESULTS / "05_biology_validation/cross_class_specificity.csv",
    RESULTS / "05_biology_validation/directed_degree_type_sign_rewiring.csv",
    RESULTS / "05_biology_validation/rewiring_coherence_tests.csv",
    RESULTS / "05_biology_validation/hallmark_c2_ora_complete_family.csv.gz",
    RESULTS / "05_biology_validation/depmap_strict_core_cross_group.csv",
    DEPMAP_MANIFEST,
]

locked_consensus_files = {}
for class_name in CFG.MULTICLASS_LABELS_NO_CONTROL:
    safe = "".join(
        char if char.isalnum() or char in "-_" else "_"
        for char in str(class_name)
    )
    path = LOCKED_CONSENSUS / f"biology_consensus_{safe}.csv"
    locked_consensus_files[str(class_name)] = path
    required_results.append(path)

missing = [str(path) for path in required_results if not Path(path).is_file()]
if missing:
    raise FileNotFoundError(
        "No se puede congelar; faltan resultados obligatorios:\n- "
        + "\n- ".join(missing)
    )

# Contrato semántico del score elegido en desarrollo.
score_lock = json.loads(LOCKED_MANIFEST.read_text(encoding="utf-8"))
if not np.isclose(float(score_lock["context_weight"]), 0.35, atol=1e-12):
    raise RuntimeError("El peso contextual bloqueado no es 0.35.")
if not np.isclose(float(score_lock["support_weight"]), 0.075, atol=1e-12):
    raise RuntimeError("El peso de soporte bloqueado no es 0.075.")
if score_lock.get("external_opened") is not False:
    raise RuntimeError(
        "El manifiesto no demuestra que el score se eligió con el externo cerrado."
    )
if score_lock.get("training_repeated") is not False:
    raise RuntimeError(
        "El manifiesto no demuestra que la reconstrucción fue post hoc y sin reentreno."
    )

# Comprobación Exact-K: las cuatro tablas deben ser no vacías, contener IDs
# únicos y conservar exactamente el mismo presupuesto de aristas.
locked_edge_counts = {}
for class_name, path in locked_consensus_files.items():
    table = pd.read_csv(path)
    if table.empty:
        raise RuntimeError(f"{class_name}: consenso biológico vacío.")
    if "candidate_edge_id" not in table.columns:
        raise RuntimeError(f"{class_name}: falta candidate_edge_id.")
    if table["candidate_edge_id"].duplicated().any():
        raise RuntimeError(f"{class_name}: candidate_edge_id duplicado.")
    locked_edge_counts[class_name] = int(len(table))
if len(set(locked_edge_counts.values())) != 1:
    raise RuntimeError(
        f"Los consensos no conservan el mismo Exact-K: {locked_edge_counts}"
    )

# DepMap debe estar completo y tampoco puede haber abierto el test externo.
depmap_manifest = json.loads(DEPMAP_MANIFEST.read_text(encoding="utf-8"))
if depmap_manifest.get("status") != "complete":
    raise RuntimeError("DepMap no está completo.")
if depmap_manifest.get("external_transcriptomic_test_opened") is not False:
    raise RuntimeError("El manifiesto DepMap no mantiene cerrado el externo.")

print("Precongelado correcto. Exact-K por clase:", locked_edge_counts)
print("Score biológico bloqueado: contexto=0.35; soporte=0.075")
print("Test externo: cerrado")


In [ ]:
paths = []
paths.extend(sorted((PROJECT_ROOT / "intergate").glob("*.py")))
paths.extend([
    PROJECT_ROOT / "configs/protocol_v6.json",
    PROJECT_ROOT / "resources_biological/PAM50_symbols.txt",
    CFG.TRAIN_EXPR_CSV, CFG.TRAIN_META_CSV,
    CFG.TEST_EXPR_CSV, CFG.TEST_META_CSV,
])
# Freeze every development result, not only the sentinel files used above.
paths.extend(sorted(path for path in RESULTS.rglob("*") if path.is_file()))
paths.extend(sorted(
    path for path in (PROJECT_ROOT / "resources_biological/msigdb").glob("*.gmt")
    if path.is_file()
))
for seed in CFG.STABILITY_SEEDS:
    seed_dir = RUN_ROOT / f"seed_{int(seed)}"
    paths.extend([
        seed_dir / "meta.json",
        seed_dir / "graph_bundle.pt",
        seed_dir / "biology_graph_bundle.pt",
        seed_dir / "model_pruned.pt",
        seed_dir / "transport_support_summary.json",
        seed_dir / "transport_adaptation_audit.csv",
    ])
    for class_name in CFG.MULTICLASS_LABELS_NO_CONTROL:
        safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in class_name)
        paths.extend([
            seed_dir / f"edges_{safe}.csv",
            seed_dir / f"biology_edges_{safe}.csv",
        ])

# Los checkpoints de las ablaciones entrenadas también forman parte del
# protocolo congelado; no basta con congelar sus tablas resumen.
ablation_manifest = pd.read_csv(
    RESULTS /
    "04_controls/trained_ablations/trained_ablation_predictive_metrics.csv"
)
for artifact_name, seed in (
    ablation_manifest.loc[~ablation_manifest.arm.eq("FULL"), [
        "artifact_name", "seed"
    ]].drop_duplicates().itertuples(index=False, name=None)
):
    ablation_dir = ARTIFACTS / str(artifact_name) / f"seed_{int(seed)}"
    paths.extend([
        ablation_dir / "meta.json",
        ablation_dir / "graph_bundle.pt",
        ablation_dir / "biology_graph_bundle.pt",
        ablation_dir / "model_pruned.pt",
        ablation_dir / "transport_support_summary.json",
        ablation_dir / "transport_adaptation_audit.csv",
    ])
    for class_name in CFG.MULTICLASS_LABELS_NO_CONTROL:
        safe = "".join(
            char if char.isalnum() or char in "-_" else "_"
            for char in class_name
        )
        paths.append(ablation_dir / f"biology_edges_{safe}.csv")
paths = [Path(path).resolve() for path in paths]
missing = [str(path) for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError("Faltan archivos congelables:\n- " + "\n- ".join(missing))

def notebook_source_sha256(path):
    notebook = json.loads(Path(path).read_text(encoding="utf-8"))
    source_only = []
    for cell in notebook.get("cells", []):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        source_only.append({
            "cell_type": cell.get("cell_type"), "source": str(source),
        })
    encoded = json.dumps(
        source_only, sort_keys=True, ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

notebook_source_hashes = {
    str(path.resolve()): notebook_source_sha256(path)
    for path in sorted((PROJECT_ROOT / "notebooks").glob("*.ipynb"))
}
payload = freeze_protocol(
    paths, LOCK,
    extra={
        "protocol_version": CFG.TRANSPORT_PROTOCOL_VERSION,
        "endpoint": list(CFG.MULTICLASS_LABELS_NO_CONTROL),
        "development_cohorts": list(CFG.TRAIN_COHORTS),
        "external_cohorts": list(CFG.TEST_COHORTS),
        "seeds": list(map(int, CFG.STABILITY_SEEDS)),
        "primary_artifact": "INTERGATE_V6",
        "external_expression_or_labels_inspected": False,
        "external_hashing_only": True,
        "single_use_external_evaluation": True,
        "biology_score_lock": {
            "status": score_lock.get("status", "locked"),
            "context_weight": 0.35,
            "support_weight": 0.075,
            "exact_k_edges_per_class": locked_edge_counts,
            "manifest": str(LOCKED_MANIFEST.resolve()),
            "consensus_directory": str(LOCKED_CONSENSUS.resolve()),
            "panels": str(LOCKED_PANELS.resolve()),
            "selected_before_biological_validation": True,
            "external_opened_at_selection": False,
            "training_repeated_for_score_selection": False,
        },
        "notebook_source_sha256_ignores_outputs": notebook_source_hashes,
    },
)
print(f"Protocolo congelado: {LOCK} ({len(payload['files'])} archivos)")
